# The Cyber Trip Wire Investigator

### Investment Banking — Banking-AI


            ## Step 0 — Notebook Header

            **Prerequisites:** Basic Python, comfort with security telemetry concepts, and familiarity with classification metrics such as precision and recall.

            **What You Will Learn:**

            - Describe how security context signals combine to determine whether an alert needs immediate escalation.
- Compare a rule-based triage baseline against a supervised model for alert prioritization.
- Construct a simple narrative handoff that explains why a cyber alert should be reviewed by a human analyst.

            **Estimated time:** 45 minutes

            **Collection context:** Investment-banking support notebooks spanning communications, cyber investigations, legal simulation, market integrity, and internal execution workflows.


## Step 1 — Investment-Banking Problem Introduction

**Why this problem matters:** Security teams lose time when analysts manually gather context that a machine could assemble before the first human joins the incident.

**Operational question:** Does this alert need urgent human escalation, and what narrative best explains the event?

**Primary stakeholders:** security operations teams, threat hunters, incident responders, and technology risk teams

**Decision supported:** Reduce triage time by combining telemetry into a faster escalation recommendation with a short investigative narrative.

**Comprehension check:** Before looking at the data, write one sentence describing which false negative would worry you more: missed exfiltration or an unnecessary wake-up call.

**Domain framing note:** This notebook is educational and synthetic. It demonstrates decision support for investment-banking and adjacent institutional workflows, not production approval or investment advice.


## Step 2 — Environment Setup

Keep the environment lightweight while preserving realistic signals such as data egress, geovelocity, and peer anomaly indicators.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, recall_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
N_ALERTS = 1500

print("Environment ready for a synthetic cyber triage workflow.")


## Step 3 — Data Creation & Context

We simulate endpoint and identity telemetry to mimic the context a SOC analyst wants immediately after a suspicious connection alert arrives.


In [ ]:
alert_df = pd.DataFrame({
    "risky_domain_score": RNG.uniform(0.0, 1.0, N_ALERTS),
    "rare_process_score": RNG.uniform(0.0, 1.0, N_ALERTS),
    "geovelocity_kmh": RNG.uniform(0, 1200, N_ALERTS),
    "off_hours_login_ratio": RNG.uniform(0.0, 1.0, N_ALERTS),
    "data_egress_mb": RNG.gamma(2.5, 160, N_ALERTS),
    "admin_tool_use_score": RNG.uniform(0.0, 1.0, N_ALERTS),
    "device_trust_score": RNG.uniform(0.1, 1.0, N_ALERTS),
    "peer_anomaly_score": RNG.uniform(0.0, 1.0, N_ALERTS),
})

escalation_signal = (
    1.2 * alert_df["risky_domain_score"]
    + 0.9 * alert_df["rare_process_score"]
    + 0.7 * alert_df["off_hours_login_ratio"]
    + 0.0015 * alert_df["data_egress_mb"]
    + 0.55 * alert_df["admin_tool_use_score"]
    + 0.8 * alert_df["peer_anomaly_score"]
    + 0.0009 * alert_df["geovelocity_kmh"]
    - 0.9 * alert_df["device_trust_score"]
    + RNG.normal(0, 0.25, N_ALERTS)
)
threshold = np.quantile(escalation_signal, 0.63)
alert_df["triage_decision"] = np.where(escalation_signal >= threshold, "Escalate", "Monitor")

print(alert_df.head(3).round(3).to_string(index=False))


## Step 4 — Exploratory Data Analysis

Look at how risky-domain scores and egress volume separate urgent cases from background noise before modeling.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.scatterplot(
    data=alert_df,
    x="risky_domain_score",
    y="data_egress_mb",
    hue="triage_decision",
    alpha=0.7,
    ax=axes[0],
)
axes[0].set_title("Cyber Alert: Domain Risk vs. Data Egress")
axes[0].set_xlabel("Risky Domain Score")
axes[0].set_ylabel("Data Egress (MB)")

sns.histplot(
    data=alert_df,
    x="peer_anomaly_score",
    hue="triage_decision",
    bins=24,
    multiple="stack",
    ax=axes[1],
)
axes[1].set_title("Peer Anomaly Distribution")
axes[1].set_xlabel("Peer Anomaly Score")

plt.tight_layout()
plt.show()
plt.close()


Alt-Text: The plots show that alerts with stronger domain risk, heavier egress, and higher peer anomaly scores cluster toward escalation, giving the later classifier an intuitive basis.


## Step 5 — Feature Engineering

We build two composite signals: one for credential abuse patterns and one for likely blast radius.


In [ ]:
analysis_df = alert_df.copy()
analysis_df["credential_abuse_index"] = (
    analysis_df["off_hours_login_ratio"]
    + analysis_df["admin_tool_use_score"]
    + analysis_df["geovelocity_kmh"] / 1200
) / 3
analysis_df["blast_radius_score"] = (
    analysis_df["data_egress_mb"] / analysis_df["data_egress_mb"].max()
    + analysis_df["peer_anomaly_score"]
    + analysis_df["risky_domain_score"]
) / 3

feature_columns = [
    "risky_domain_score",
    "rare_process_score",
    "geovelocity_kmh",
    "off_hours_login_ratio",
    "data_egress_mb",
    "admin_tool_use_score",
    "device_trust_score",
    "peer_anomaly_score",
    "credential_abuse_index",
    "blast_radius_score",
]
print(analysis_df[feature_columns].head(3).round(3).to_string(index=False))


## Step 6 — Baseline Establishment

A simple SOC baseline is a threshold rule: escalate when the domain is very risky or when data egress spikes above a fixed cut-off.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    analysis_df[feature_columns],
    analysis_df["triage_decision"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=analysis_df["triage_decision"],
)

baseline_pred = np.where(
    (X_test["risky_domain_score"] >= 0.78) | (X_test["data_egress_mb"] >= 520),
    "Escalate",
    "Monitor",
)

print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")
print(f"Baseline recall: {recall_score(y_test, baseline_pred, pos_label='Escalate'):.3f}")


## Step 7 — Model Training & Evaluation

Train a stronger classifier and compare how much escalation recall improves versus the simple threshold rule.


In [ ]:
triage_model = RandomForestClassifier(
    n_estimators=240,
    min_samples_leaf=5,
    random_state=RANDOM_SEED,
)
triage_model.fit(X_train, y_train)
model_pred = triage_model.predict(X_test)

print(f"Model accuracy: {accuracy_score(y_test, model_pred):.3f}")
print(f"Model recall: {recall_score(y_test, model_pred, pos_label='Escalate'):.3f}")
print(classification_report(y_test, model_pred))


## Step 8 — Interpretability & Explainability

Feature importance makes the automated narrative defensible: the analyst should know whether the system escalated because of blast radius, identity signals, or both.


In [ ]:
importance_frame = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": triage_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

sns.barplot(data=importance_frame, y="feature", x="importance", color="#4C78A8")
plt.title("Key Drivers of Cyber Escalation")
plt.xlabel("Model Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()
plt.close()

print(importance_frame.head(6).round(3).to_string(index=False))


## Step 9 — Operational Application

Turn the model output into a human-readable alert narrative so an analyst gets context before they start investigating.


In [ ]:
scenario_df = pd.DataFrame(
    {
        "risky_domain_score": [0.92, 0.38, 0.71],
        "rare_process_score": [0.86, 0.24, 0.61],
        "geovelocity_kmh": [930, 120, 410],
        "off_hours_login_ratio": [0.88, 0.15, 0.64],
        "data_egress_mb": [820, 90, 360],
        "admin_tool_use_score": [0.73, 0.12, 0.58],
        "device_trust_score": [0.22, 0.91, 0.44],
        "peer_anomaly_score": [0.89, 0.18, 0.63],
    },
    index=["wake_up_now", "likely_noise", "needs_review"],
)
scenario_df["credential_abuse_index"] = (
    scenario_df["off_hours_login_ratio"]
    + scenario_df["admin_tool_use_score"]
    + scenario_df["geovelocity_kmh"] / 1200
) / 3
scenario_df["blast_radius_score"] = (
    scenario_df["data_egress_mb"] / analysis_df["data_egress_mb"].max()
    + scenario_df["peer_anomaly_score"]
    + scenario_df["risky_domain_score"]
) / 3

scenario_df["decision"] = triage_model.predict(scenario_df[feature_columns])
for case_name, row in scenario_df.iterrows():
    narrative = (
        f"{case_name}: {row['decision']} because risky_domain={row['risky_domain_score']:.2f}, "
        f"egress={row['data_egress_mb']:.0f}MB, off_hours={row['off_hours_login_ratio']:.2f}, "
        f"device_trust={row['device_trust_score']:.2f}."
    )
    print(narrative)


            ## Step 10 — Summary & Reflection

            **What you accomplished:** You built a synthetic cyber triage workflow that compresses telemetry into a faster escalation decision and a short analyst-ready narrative.

            **Limitations to note:**

            - The telemetry is synthetic and does not capture sequence timing, host lineage, or threat-intelligence enrichment.
- Narratives are assembled from a small set of fields rather than a true case graph.
- Real security deployment would require validation against production logging quality and response playbooks.

            **Ethical reflection:** Automated triage can reduce fatigue, but it can also normalize intrusive monitoring if organizations do not set clear limits on data use and retention.

            **Reflection questions:**

            1. Which additional telemetry source would most improve confidence in the escalation narrative?
2. How would you change the metrics if the security team cared more about analyst fatigue than recall?
3. What evidence should always trigger human review even if the model says `Monitor`?


            ## References

            - Scikit-learn user guide: ensemble methods and classification metrics.
- Public incident-response and SOC triage practices from vendor-neutral security operations guidance.
- MITRE ATT&CK concepts for contextualizing suspicious behavior patterns.
